# EXP0000_SMOKE — Kaggle GPU Diagnostic Smoke Test & Checkpoint Recovery Drill (`STEP-027` / Gate `G09`)
**উদ্দেশ্য:** আসল `REL0001` (শুধুমাত্র `Train` ও `Val` slice — **Test সম্পূর্ণ সিলড!**) ব্যবহার করে Kaggle GPU (`T4 x1` বা `P100`) পরিবেশে যাচাই করা:
1. **XGBoost (`MC0004`) + TreeSHAP** Additivity & Validation AUROC,
2. **4-Modality PyTorch `CVDMMTransformer` (`MC0015`)** (`d_model=64`, `~ModalityMask_CELS`, zero `NaN` across `DS0001–DS0004`),
3. **Deliberate Interruption (`Epoch 2`) -> SHA-256 Verify -> Full State Restore (`model + optimizer + scheduler + GradScaler + CPU/CUDA RNG`) -> Exact Resume (`Epoch 4`)**,
4. **Diagnostic Neural `KernelExplainer` SHAP Additivity & Missing-Modality Zero-`NaN` Stress Check**.

In [ ]:
# Cell 1: Kaggle Environment Lock Check, Seeds & REL0001 Auto-Discovery (Works with folders OR flat files; Test Locked)
import os
import json
import math
import random
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

TEST_SEALED = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_DIR = Path('/kaggle/working/EXP0000_SMOKE') if Path('/kaggle').exists() else Path('08_Checkpoints/binaries/EXP0000_SMOKE_KAGGLE')
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

TRAIN_SHARD = VAL_SHARD = None
for root_cand in [Path('/kaggle/input'), Path('05_Preprocessing_Alignment/processed/REL0001'), Path('.')]:
    if not root_cand.exists():
        continue
    tr_hits = list(root_cand.rglob('rel0001_train.jsonl'))
    va_hits = list(root_cand.rglob('rel0001_val.jsonl'))
    if tr_hits and va_hits:
        TRAIN_SHARD, VAL_SHARD = tr_hits[0], va_hits[0]
        break

if TRAIN_SHARD is None or VAL_SHARD is None:
    raise FileNotFoundError('Attach REL0001 (rel0001_train.jsonl & rel0001_val.jsonl) via + Add Input first!')

print(f'Device={DEVICE} | PyTorch={torch.__version__} | SHAP={shap.__version__}')
print(f'TRAIN_SHARD={TRAIN_SHARD}')
print(f'VAL_SHARD={VAL_SHARD} | TEST_SEALED={TEST_SEALED}')

In [ ]:
# Cell 2: Load Stratified REL0001 Smoke Slices (Strictly Train & Val Only — Zero Test Reads)
COHORT_MODALITY_SLICES = {
    'DS0001': {'clinical': (0,1,2,3,4,5,6,7,8,9,10), 'laboratory': (), 'lifestyle': ()},
    'DS0002': {'clinical': (0,1,2,3,4,5,6,7,12,13,14), 'laboratory': (8,9,10,11), 'lifestyle': ()},
    'DS0003': {'clinical': (0,1,2,3,4,5,6,7,8,9,10,11,24,25,26), 'laboratory': (12,13,14,15,16,17,18,19), 'lifestyle': (20,21,22,23,27,28,29,30)},
    'DS0004': {'clinical': (0,1,2,3,4,5,6,7,8,9,10,11,22,23,24), 'laboratory': (12,13,14,15), 'lifestyle': (16,17,18,19,20,21)},
}
COHORT_MODALITY_DIMS = {ds: {m: len(ix) for m, ix in mods.items()} for ds, mods in COHORT_MODALITY_SLICES.items()}

def load_stratified_smoke(partition: str, per_cohort: int = 150):
    if partition.lower() == 'test' and TEST_SEALED:
        raise PermissionError('TEST_PARTITION_SEALED: Final Test shard is locked until Gate G11 PASS!')
    shard = TRAIN_SHARD if partition == 'train' else VAL_SHARD
    buckets = {ds: [] for ds in COHORT_MODALITY_SLICES}
    with open(shard, encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            ds = rec['DS']
            if ds in buckets and len(buckets[ds]) < per_cohort:
                buckets[ds].append(rec)
            if all(len(v) >= per_cohort for v in buckets.values()):
                break
    out = []
    for ds in sorted(buckets):
        out.extend(buckets[ds])
    return out

smoke_train = load_stratified_smoke('train', per_cohort=150)
smoke_val = load_stratified_smoke('val', per_cohort=60)
print('Train smoke counts:', pd.Series([r['DS'] for r in smoke_train]).value_counts().to_dict())
print('Val   smoke counts:', pd.Series([r['DS'] for r in smoke_val]).value_counts().to_dict())

In [ ]:
# Cell 3: Smoke Check 1 — MC0004 XGBoost Baseline + Exact TreeSHAP Additivity Check
def align_tabular_matrix(recs, width=39):
    ds_map = {'DS0001': 0, 'DS0002': 1, 'DS0003': 2, 'DS0004': 3}
    raw_len = width - 8
    X = np.zeros((len(recs), width), dtype=np.float32)
    y = np.zeros(len(recs), dtype=np.int64)
    for i, r in enumerate(recs):
        X[i, :4] = r['ModalityMask_CELS']
        X[i, 4 + ds_map[r['DS']]] = 1.0
        tf = r['TabularFeatures'][:raw_len]
        X[i, 8:8+len(tf)] = tf
        y[i] = int(r['Target_y'])
    return X, y

X_tr, y_tr = align_tabular_matrix(smoke_train)
X_va, y_va = align_tabular_matrix(smoke_val)
pos_w = float((y_tr == 0).sum()) / max(1.0, float((y_tr == 1).sum()))

xgb_smoke = XGBClassifier(
    n_estimators=60, max_depth=4, learning_rate=0.08, subsample=0.8,
    scale_pos_weight=pos_w, eval_metric='auc', random_state=42, n_jobs=-1
)
xgb_smoke.fit(X_tr, y_tr)
val_probs_xgb = xgb_smoke.predict_proba(X_va)[:, 1]
xgb_val_auc = roc_auc_score(y_va, val_probs_xgb)

explainer = shap.TreeExplainer(xgb_smoke)
phi_xgb = np.asarray(explainer.shap_values(X_va[:32]))
raw_margins = xgb_smoke.predict(X_va[:32], output_margin=True)
base_xgb = float(np.asarray(explainer.expected_value).reshape(-1)[0])
max_add_err_xgb = float(np.max(np.abs(raw_margins - (base_xgb + phi_xgb.sum(axis=1)))))
print(f'XGB Smoke Val AUROC={xgb_val_auc:.4f} | TreeSHAP Max Additivity Error={max_add_err_xgb:.6e} (PASS <= 0.02: {max_add_err_xgb <= 0.02})')

In [ ]:
# Cell 4: Smoke Check 2 — 4-Modality CVDMMTransformer (PyTorch GPU) + Deliberate Interrupt/Resume Recovery
class SEBlock1D(nn.Module):
    def __init__(self, c: int):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(c, max(4, c // 4)), nn.SiLU(), nn.Linear(max(4, c // 4), c), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(dim=-1)).unsqueeze(-1)

class CompactResNet1D(nn.Module):
    def __init__(self, in_leads: int = 12, d_model: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_leads, 32, 7, stride=4, padding=3, bias=False), nn.BatchNorm1d(32), nn.GELU(),
            nn.Conv1d(32, d_model, 5, stride=4, padding=2, bias=False), nn.BatchNorm1d(d_model), nn.GELU(), SEBlock1D(d_model)
        )
    def forward(self, x):
        return self.net(x).mean(dim=-1)

class TabularModalityMLP(nn.Module):
    def __init__(self, in_dim: int, d_model: int = 64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(0.15), nn.Linear(d_model, d_model))
    def forward(self, x):
        return self.net(x)

class CVDMMTransformer(nn.Module):
    def __init__(self, cohort_dims: dict, d_model: int = 64, nhead: int = 4, n_layers: int = 2):
        super().__init__()
        self.d_model = d_model
        self.tab_adapters = nn.ModuleDict({f'{ds}__{m}': TabularModalityMLP(dim, d_model) for ds, mods in cohort_dims.items() for m, dim in mods.items() if dim > 0})
        self.ecg_encoder = CompactResNet1D(12, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.mod_type_emb = nn.Parameter(torch.randn(1, 4, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.GELU(), nn.Linear(32, 1))

    def forward(self, ds_id: str, tab_mods: dict, mask_cels: torch.Tensor, ecg_1d: torch.Tensor | None = None):
        b, dev = mask_cels.size(0), mask_cels.device
        toks = []
        for idx, mod in enumerate(['clinical', 'ecg', 'laboratory', 'lifestyle']):
            if mod == 'ecg':
                emb = self.ecg_encoder(ecg_1d) if ecg_1d is not None else torch.zeros(b, self.d_model, device=dev)
            else:
                key = f'{ds_id}__{mod}'
                emb = self.tab_adapters[key](tab_mods[mod]) if (key in self.tab_adapters and mod in tab_mods) else torch.zeros(b, self.d_model, device=dev)
            toks.append(emb + self.mod_type_emb[:, idx, :])
        seq = torch.cat([self.cls_token.expand(b, -1, -1), torch.stack(toks, dim=1)], dim=1)
        pad_mask = torch.cat([torch.zeros(b, 1, dtype=torch.bool, device=dev), ~mask_cels.bool()], dim=1)
        return self.head(self.norm(self.encoder(seq, src_key_padding_mask=pad_mask)))[:, 0].squeeze(-1)

def run_epoch(model, opt, sched, scaler, recs, ds_id):
    model.train()
    slices = COHORT_MODALITY_SLICES[ds_id]
    sub = [r for r in recs if r['DS'] == ds_id]
    X = np.asarray([r['TabularFeatures'] for r in sub], dtype=np.float32)
    mods = {m: torch.from_numpy(X[:, list(ix)]).to(DEVICE) for m, ix in slices.items() if len(ix) > 0}
    mask = torch.tensor([r['ModalityMask_CELS'] for r in sub], dtype=torch.bool, device=DEVICE)
    y = torch.tensor([r['Target_y'] for r in sub], dtype=torch.float32, device=DEVICE)
    ecg_dummy = torch.randn(len(sub), 12, 500, device=DEVICE) * 0.1 if ds_id in ('DS0001', 'DS0002') else None
    opt.zero_grad(set_to_none=True)
    with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
        logits = model(ds_id, mods, mask, ecg_1d=ecg_dummy)
        loss = F.binary_cross_entropy_with_logits(logits, y)
    assert torch.isfinite(loss), f'Non-finite loss for {ds_id}!'
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    sched.step()
    return float(loss.item())

seed_everything(42)
model_a = CVDMMTransformer(COHORT_MODALITY_DIMS).to(DEVICE)
opt_a = torch.optim.AdamW(model_a.parameters(), lr=1e-3, weight_decay=1e-4)
sched_a = torch.optim.lr_scheduler.CosineAnnealingLR(opt_a, T_max=16)
scaler_a = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

# Epochs 1-2 -> DELIBERATE INTERRUPT -> Full Checkpoint Save
losses_phase1 = []
for ep in range(1, 3):
    for ds in ['DS0001', 'DS0002', 'DS0003', 'DS0004']:
        losses_phase1.append((ep, ds, run_epoch(model_a, opt_a, sched_a, scaler_a, smoke_train, ds)))

ckpt_pt = SMOKE_DIR / 'CKPT0001_SMOKE_interrupt.pt'
torch.save({
    'epoch': 2,
    'model_state_dict': model_a.state_dict(),
    'optimizer_state_dict': opt_a.state_dict(),
    'scheduler_state_dict': sched_a.state_dict(),
    'scaler_state_dict': scaler_a.state_dict(),
    'rng_python': random.getstate(),
    'rng_numpy': np.random.get_state(),
    'rng_torch': torch.get_rng_state(),
    'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
}, ckpt_pt)
ckpt_sha = hashlib.sha256(ckpt_pt.read_bytes()).hexdigest()
print(f'Interrupted after Epoch 2 | Saved {ckpt_pt.name} | SHA256={ckpt_sha}')

# Verify SHA-256 -> Full State Restore -> Resume Epochs 3-4
assert hashlib.sha256(ckpt_pt.read_bytes()).hexdigest() == ckpt_sha
bundle = torch.load(ckpt_pt, map_location=DEVICE, weights_only=False)
model_b = CVDMMTransformer(COHORT_MODALITY_DIMS).to(DEVICE)
opt_b = torch.optim.AdamW(model_b.parameters(), lr=1e-3, weight_decay=1e-4)
sched_b = torch.optim.lr_scheduler.CosineAnnealingLR(opt_b, T_max=16)
scaler_b = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

model_b.load_state_dict(bundle['model_state_dict'])
opt_b.load_state_dict(bundle['optimizer_state_dict'])
sched_b.load_state_dict(bundle['scheduler_state_dict'])
scaler_b.load_state_dict(bundle['scaler_state_dict'])
random.setstate(bundle['rng_python'])
np.random.set_state(bundle['rng_numpy'])
torch.set_rng_state(bundle['rng_torch'].cpu())
if torch.cuda.is_available() and bundle.get('rng_cuda') is not None:
    torch.cuda.set_rng_state_all([t.cpu() for t in bundle['rng_cuda']])

losses_phase2 = []
for ep in range(3, 5):
    for ds in ['DS0001', 'DS0002', 'DS0003', 'DS0004']:
        losses_phase2.append((ep, ds, run_epoch(model_b, opt_b, sched_b, scaler_b, smoke_train, ds)))
print('Phase 2 resumed finite losses:', [(e, d, round(l, 4)) for e, d, l in losses_phase2])

In [ ]:
# Cell 5: Smoke Check 3 — Neural KernelSHAP Additivity, Missing-Modality Zero-NaN & G09 Report Export
model_b.eval()
ds4_tr = [r for r in smoke_train if r['DS'] == 'DS0004']
ds4_va = [r for r in smoke_val if r['DS'] == 'DS0004'][:16]
X_tr4 = np.asarray([r['TabularFeatures'] for r in ds4_tr], dtype=np.float32)
X_va4 = np.asarray([r['TabularFeatures'] for r in ds4_va], dtype=np.float32)
bg4 = X_tr4[:20]

def ds4_predict_prob(X_arr: np.ndarray) -> np.ndarray:
    slices = COHORT_MODALITY_SLICES['DS0004']
    with torch.no_grad():
        mods = {m: torch.from_numpy(X_arr[:, list(ix)]).float().to(DEVICE) for m, ix in slices.items() if len(ix) > 0}
        masks = torch.tensor([[1, 0, 1, 1]] * len(X_arr), dtype=torch.bool, device=DEVICE)
        return torch.sigmoid(model_b('DS0004', mods, masks)).cpu().numpy()

k_exp = shap.KernelExplainer(ds4_predict_prob, bg4)
phi_nn = np.asarray(k_exp.shap_values(X_va4[:5], nsamples=200, l1_reg='auto', silent=True))
preds_nn = ds4_predict_prob(X_va4[:5])
base_nn = float(np.asarray(k_exp.expected_value).reshape(-1)[0])
max_nn_add_err = float(np.max(np.abs(preds_nn - (base_nn + phi_nn.sum(axis=1)))))

# Missing-modality stress checks (1, 2, 3 modalities masked out)
with torch.no_grad():
    mods4 = {m: torch.from_numpy(X_va4[:4, list(ix)]).float().to(DEVICE) for m, ix in COHORT_MODALITY_SLICES['DS0004'].items() if len(ix) > 0}
    for bits in [[1,0,1,1], [1,0,0,1], [1,0,0,0]]:
        msk = torch.tensor([bits] * 4, dtype=torch.bool, device=DEVICE)
        lg = model_b('DS0004', mods4, msk)
        assert torch.isfinite(lg).all().item(), f'Non-finite logits for mask {bits}'

report = {
    'status': 'KAGGLE_SMOKE_G09_READY',
    'experiment_id': 'EXP0000_SMOKE',
    'device': str(DEVICE),
    'test_partition_accessed': False,
    'xgb_smoke_val_roc_auc': round(float(xgb_val_auc), 6),
    'xgb_treeshap_max_additivity_error': max_add_err_xgb,
    'cvd_mm_transformer_param_count': sum(p.numel() for p in model_b.parameters() if p.requires_grad),
    'checkpoint_path': str(ckpt_pt),
    'checkpoint_sha256': ckpt_sha,
    'neural_kernelshap_max_additivity_error': max_nn_add_err,
    'missing_masks_zero_nan': True,
    'resumed_losses_finite': all(math.isfinite(l) for _, _, l in losses_phase2),
}
rep_file = SMOKE_DIR / 'smoke_g09_report.json'
rep_file.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))